In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class ModelArgs:
    # 基础维度
    hidden_size: int = 2048
    ffn_hidden_size: int = 4096 # 用于非 MoE 层或 Dense 层，MoE 层用下面的参数
    
    # MoE 核心参数
    num_experts: int = 64
    moe_ffn_hidden_size: int = 768  # 路由专家维度
    moe_router_topk: int = 4
    moe_shared_expert_intermediate_size: int = 768 # 共享专家维度
    
    # Router 机制参数
    moe_router_score_function: str = 'sigmoid' # 关键点：使用 Sigmoid
    moe_router_enable_expert_bias: bool = True
    moe_router_bias_update_rate: float = 0.001
    moe_router_topk_scaling_factor: float = 1.0 # 有些实现中会用
    
    # 激活函数与初始化
    swiglu: bool = True
    disable_bias_linear: bool = True # FFN 内部线性层无 Bias
    init_method_std: float = 0.02

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def silu(x):
    return x / (1 + np.exp(-x))

x = np.linspace(-5, 5, 1200)
y = silu(x)

# ===== 正方形画布 =====
fig, ax = plt.subplots(figsize=(8, 8))

# 曲线
ax.plot(x, y, color="#1f77b4", linewidth=3.0, zorder=3)

# ===== 数值范围（你的最新要求）=====
ax.set_xlim(-3, 3)
ax.set_ylim(-1, 5)

# ===== 中心轴 =====
ax.spines["left"].set_position("zero")
ax.spines["bottom"].set_position("zero")
ax.spines["left"].set_linewidth(1.4)
ax.spines["bottom"].set_linewidth(1.4)
ax.spines["left"].set_color("#222222")
ax.spines["bottom"].set_color("#222222")

# ===== 去掉外框 =====
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ===== 虚线网格 =====
ax.set_xticks(np.arange(-3, 4, 1))
ax.set_yticks(np.arange(-1, 6, 1))
ax.grid(True, linestyle="--", linewidth=0.8, color="#d9d9d9", alpha=0.8)

# 去掉所有刻度与文字
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.tick_params(length=0)

# 坐标比例严格一致
ax.set_aspect("equal", adjustable="box")

# 留白，避免 tight 破坏包裹感
fig.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.92)

# 保存为 SVG
plt.savefig("silu.svg", format="svg")
plt.close()

In [ ]:
# 使用提供的 MODEL_ARGS 初始化配置
args = ModelArgs(
    hidden_size=2048,
    num_experts=64,
    moe_ffn_hidden_size=768,       # 细粒度专家
    moe_shared_expert_intermediate_size=768, # 共享专家
    moe_router_topk=4,
    moe_router_score_function='sigmoid',
    moe_router_enable_expert_bias=True,
    moe_router_bias_update_rate=0.001
)

print(f"DeepSeek V3 Config: {args}")

model = DeepSeekV3MoELayer(args)

# 模拟输入 (Batch=2, Seq=128, Dim=2048)
x = torch.randn(2, 128, 2048)

output = model(x)

print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

# 验证 Sigmoid Router
print("\nCheck Router Logic:")
# 提取 router 内部的 sigmoid 行为
logits = model.router.gate(F.normalize(x.view(-1, 2048), dim=-1))
scores = torch.sigmoid(logits)
print(f"Max Score (Sigmoid): {scores.max().item():.4f} (Should be <= 1.0)")
print(f"Min Score (Sigmoid): {scores.min().item():.4f} (Should be >= 0.0)")

DeepSeek V3 Config: ModelArgs(hidden_size=2048, ffn_hidden_size=4096, num_experts=64, moe_ffn_hidden_size=768, moe_router_topk=4, moe_shared_expert_intermediate_size=768, moe_router_score_function='sigmoid', moe_router_enable_expert_bias=True, moe_router_bias_update_rate=0.001, moe_router_topk_scaling_factor=1.0, swiglu=True, disable_bias_linear=True, init_method_std=0.02)

Input shape: torch.Size([2, 128, 2048])
Output shape: torch.Size([2, 128, 2048])

Check Router Logic:
Max Score (Sigmoid): 0.5165 (Should be <= 1.0)
Min Score (Sigmoid): 0.4861 (Should be >= 0.0)


: 